# Single-carrier label questions on the twelve-carrier message, Kaggle T4

Same Tesla T4, same Qwen3-4B revision, FP16, greedy decoding. Do not switch GPUs.

Do **not** regenerate A. Do **not** overwrite `results_stage_plan4_recon12`, timing, hold or confirm. No adapter, no new compression method.

All twelve carriers' result states stay in every message. B keeps the same private facts (shipment request, prices, budget). Only the request changes:

> According to Agent A's results, is Carrier Q eligible or ineligible? Reply with exactly eligible or ineligible.

Four conditions, every carrier in every case: 9 cases x 12 questions x 4 conditions = 432 short answers.
1. **generic_r**: generic support + transferred R
2. **generic_fresh_r**: the same support + R recomputed after it
3. **full_kv**: rebuilt full sender cache
4. **text_result**: RESULT lines as text

Each question starts from an independent copy of its message cache; the run checks the stored cache is unchanged after each condition's twelve questions. Answers must be exactly `eligible` or `ineligible` (case, one trailing period and wrapping quotes/asterisks tolerated; `exact_raw` records strict matches).

**Cell 4** (`--phase check`) rebuilds the caches and checks them against the recon12 construction summary. **Cell 5** (`--phase compare`) asks the 432 questions.

Upload `src/`, `configs/`, `scripts/`, `data/plan4_timing12.jsonl`, and from `results_stage_plan4_recon12/`: `a_artifacts_12.jsonl`, `evaluations.jsonl`, `check_summary.json`.

Download `/kaggle/working/results_stage_plan4_query12`.

In [ ]:
import os, json, shutil, sys
from pathlib import Path

os.environ["LATENT_RELAY_INFER_DTYPE"] = "fp16"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()

candidates = [Path.cwd()]
if Path("/kaggle/input").is_dir():
    for root, dirs, files in os.walk("/kaggle/input"):
        dirs[:] = [d for d in dirs if d not in {".git", ".pytest_cache"}]
        if "stage_plan4_query12.py" in files and Path(root).name == "src":
            candidates.append(Path(root).parent)

REPO = None
for cand in candidates:
    if (cand / "src" / "stage_plan4_query12.py").is_file() and (cand / "data" / "plan4_timing12.jsonl").is_file():
        REPO = cand
        break
if REPO is None:
    raise FileNotFoundError("Need src/stage_plan4_query12.py and data/plan4_timing12.jsonl.")

if ON_KAGGLE:
    dest = WORK / "latent-relay-audit"
    if str(REPO).startswith("/kaggle/input") or not (dest / "src" / "stage_plan4_query12.py").is_file():
        shutil.copytree(REPO, dest, dirs_exist_ok=True)
    REPO = dest
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
recon = REPO / "results_stage_plan4_recon12"
print("REPO", REPO.resolve())
for name in ("a_artifacts_12.jsonl", "evaluations.jsonl", "check_summary.json"):
    print(name, (recon / name).is_file())

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers==4.57.3", "tokenizers==0.22.2", "huggingface_hub==0.36.2",
    "safetensors==0.7.0", "accelerate==1.13.0", "datasets==4.4.1",
])
import torch, transformers
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("cuda", torch.cuda.is_available(), gpu)
print("transformers", transformers.__version__)
assert transformers.__version__.startswith("4.57")
if gpu is not None and "T4" not in gpu:
    raise RuntimeError(f"Stay on the locked Tesla T4. Got {gpu}.")

In [ ]:
subprocess.check_call([sys.executable, "scripts/fetch_upstream.py"])
print("Dest /kaggle/working/results_stage_plan4_query12. Check first, then compare.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_query12", "--split", "plan4_timing12", "--phase", "check"])
print("query12 check exit", rc)
path = WORK / "results_stage_plan4_query12" / "check_summary.json"
if not path.is_file():
    path = REPO / "results" / "stage_plan4_query12" / "run" / "check_summary.json"
check = json.loads(path.read_text(encoding="utf-8"))
print(json.dumps({"n": check.get("n"), "all_passed": check.get("all_passed")}, indent=2))
if not check.get("all_passed"):
    raise RuntimeError("Construction does not match recon12. Do not run compare.")

In [ ]:
rc = subprocess.call([sys.executable, "-m", "src.stage_plan4_query12", "--split", "plan4_timing12", "--phase", "compare"])
print("query12 compare exit", rc)

In [ ]:
src = WORK / "results_stage_plan4_query12"
if not (src / "summary.json").is_file():
    src = REPO / "results" / "stage_plan4_query12" / "run"
data = json.loads((src / "summary.json").read_text(encoding="utf-8"))
keys = ("n", "parse_ok", "exact_raw", "truncated", "correct", "ineligible_marked_eligible",
        "eligible_marked_ineligible", "unparsed", "cases_all_twelve_correct")
print("rows", data["n_rows"], "of", data["n_expected"])
for cond, block in data["receiver"].items():
    print(cond, {k: block.get(k) for k in keys})
print("\nquery vs twelve-line, per condition")
for cond, block in data["query_vs_twelve_line"].items():
    print(cond, block["crosstab"])
print("\npaired by case (labels correct)")
for name, block in data["paired_by_case"].items():
    c = block["correct"]
    print(name, {k: c.get(k) for k in ("total_delta", "mean_delta_per_case", "ci95_per_case", "cases_increased", "cases_decreased")})